# Day 5 — Checkpoints and Data Docs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/great-expectations-certified/notebooks/day-05-checkpoints-data-docs.ipynb)

**Badge:** Practice  
**Course:** Great Expectations for Data Quality

---

## What you will learn

- The difference between a `Checkpoint`, a `ValidationDefinition`, and a `CheckpointResult`
- How to create a Checkpoint that runs multiple suites in a single call
- How to inspect a `CheckpointResult` object: `.success`, `.run_results`
- How `UpdateDataDocsAction` triggers HTML report generation
- How to write a CI gate that exits non-zero on any validation failure
- How to customize Data Docs sites

> **Tip:** `CheckpointResult.success` is `False` if *any* suite in the Checkpoint fails. A simple `if not result.success: sys.exit(1)` is the correct CI gate pattern.

## Concept: Checkpoint vs ValidationDefinition vs CheckpointResult

Read the official guide: [Checkpoints](https://docs.greatexpectations.io/docs/core/validate_data/checkpoints/)

| Object | Role |
|---|---|
| `ValidationDefinition` | Pairs *one suite* with *one batch definition*. The atomic unit of validation. |
| `Checkpoint` | Orchestrates *many* `ValidationDefinition`s and fires *Actions* on completion. |
| `CheckpointResult` | The immutable result object returned by `checkpoint.run()`. Contains per-suite pass/fail and aggregate `.success`. |
| `Action` | Post-validation side-effect: update Data Docs, send Slack alert, etc. |

The layered design means you can reuse a `ValidationDefinition` across multiple Checkpoints — one for CI (strict), one for monitoring (lenient).

## Install dependencies

In [ ]:
%pip install great_expectations --quiet

## Setup: Rebuild Context, Suites, and Batch from Day 4

Each notebook is self-contained. We re-create the suites and data source here.

In [ ]:
import great_expectations as gx
import pandas as pd
import numpy as np
import sys

context = gx.get_context()   # EphemeralDataContext

# ---- Synthetic orders data ----
rng = np.random.default_rng(42)
N = 200
orders_df = pd.DataFrame({
    'order_id':    [f'ORD-{i:05d}' for i in range(1, N + 1)],
    'customer_id': rng.integers(1000, 9999, size=N),
    'amount':      rng.uniform(5.0, 500.0, size=N).round(2),
    'status':      rng.choice(['pending', 'shipped', 'delivered', 'cancelled'], size=N),
    'email':       [f'user{i}@example.com' for i in range(1, N + 1)],
})

# ---- Data source + asset + batch definition ----
data_source = context.data_sources.add_pandas('orders_source')
data_asset  = data_source.add_dataframe_asset('orders_asset')
batch_def   = data_asset.add_batch_definition_whole_dataframe('orders_batch')

print('Context ready. Rows:', len(orders_df))

In [ ]:
# ---- Recreate both suites ----
suite_critical = context.suites.add(
    gx.ExpectationSuite(name='orders.critical')
)
suite_critical.add_expectation(gx.expectations.ExpectTableRowCountToBeBetween(min_value=50, max_value=1_000_000))
for col in ['order_id', 'customer_id', 'amount', 'status', 'email']:
    suite_critical.add_expectation(gx.expectations.ExpectColumnToExist(column=col))
for col in ['order_id', 'customer_id', 'amount']:
    suite_critical.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column=col))
context.suites.save(suite_critical)

suite_dist = context.suites.add(
    gx.ExpectationSuite(name='orders.distributions')
)
suite_dist.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(column='amount', min_value=0.01, max_value=10_000.0)
)
suite_dist.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column='status', value_set=['pending', 'shipped', 'delivered', 'cancelled']
    )
)
suite_dist.add_expectation(
    gx.expectations.ExpectColumnValuesToMatchRegex(
        column='email', regex=r'^[\w.+-]+@[\w-]+\.[a-zA-Z]{2,}$'
    )
)
context.suites.save(suite_dist)

print('Suites created: orders.critical, orders.distributions')

## 1. Creating Validation Definitions

A `ValidationDefinition` is the atomic pairing of one batch definition and one suite. We need one per suite before we can build a Checkpoint.

In [ ]:
vd_critical = context.validation_definitions.add(
    gx.ValidationDefinition(
        name='vd_orders_critical',
        data=batch_def,
        suite=context.suites.get('orders.critical'),
    )
)

vd_dist = context.validation_definitions.add(
    gx.ValidationDefinition(
        name='vd_orders_distributions',
        data=batch_def,
        suite=context.suites.get('orders.distributions'),
    )
)

print('ValidationDefinitions created:', [vd_critical.name, vd_dist.name])

## 2. Creating a Checkpoint that Runs Both Suites

A `Checkpoint` takes a list of `ValidationDefinition`s and an optional list of `Actions`. Running the Checkpoint validates all definitions and fires all actions.

See [Checkpoint docs](https://docs.greatexpectations.io/docs/core/validate_data/checkpoints/) and [Actions docs](https://docs.greatexpectations.io/docs/core/validate_data/actions/).

In [ ]:
# UpdateDataDocsAction regenerates HTML after each run
update_docs_action = gx.checkpoint.UpdateDataDocsAction(name='update_data_docs')

checkpoint = context.checkpoints.add(
    gx.Checkpoint(
        name='orders_checkpoint',
        validation_definitions=[vd_critical, vd_dist],
        actions=[update_docs_action],
        result_format={'result_format': 'COMPLETE'},
    )
)

print('Checkpoint created:', checkpoint.name)
print('Validation definitions:', [vd.name for vd in checkpoint.validation_definitions])

## 3. Running the Checkpoint and Inspecting the Result

After running, we inspect the `CheckpointResult` object:
- `.success` — aggregate pass/fail across all suites
- `.run_results` — dict keyed by validation result identifier, each containing per-suite results

In [ ]:
checkpoint_result = checkpoint.run(
    batch_parameters={'dataframe': orders_df}
)

# Top-level success
print('CheckpointResult.success:', checkpoint_result.success)
print()

# Iterate run_results: each key is a ValidationResultIdentifier
for identifier, vr in checkpoint_result.run_results.items():
    suite_name = identifier.expectation_suite_identifier.name
    suite_success = vr.success
    evaluated = len(vr.results)
    failed = sum(1 for r in vr.results if not r.success)
    print(f'Suite: {suite_name!r:30s}  success={suite_success}  evaluated={evaluated}  failed={failed}')

## 4. What Data Docs Contains

After `UpdateDataDocsAction` fires, GX generates static HTML in the configured Data Docs site directory. The generated pages include:

- **Index page** (`index.html`) — lists all runs with timestamp, suite name, and pass/fail badge.
- **Validation result page** — per-expectation rows: expectation type, observed value, success/failure, unexpected samples.
- **Expectation Suite page** — the suite definition formatted as human-readable documentation, with descriptions from `_prescriptive_renderer` (Day 7).

In an `EphemeralDataContext` the HTML is generated in a temp directory. In a `FileDataContext` it lives under `great_expectations/uncommitted/data_docs/`.

Run `context.open_data_docs()` in a local environment to open the generated site in your browser.

In [ ]:
# Inspect the Data Docs site config
docs_sites = context.variables.data_docs_sites if hasattr(context, 'variables') else {}
print('Data Docs sites configured:', list(docs_sites.keys()) if docs_sites else 'default (local_site)')

# In Colab we cannot open a browser, but we can describe what was built
print()
print('After UpdateDataDocsAction, GX generated:')
print('  - index.html                (run history with pass/fail badges)')
print('  - validations/<run>/<suite>.html  (per-expectation rows + unexpected samples)')
print('  - expectations/<suite>.html      (suite definition as human-readable docs)')
print()
print('To open locally: context.open_data_docs()')

## 5. CI Gate Function

The standard CI gate is one line: `if not checkpoint_result.success: sys.exit(1)`. We wrap it in a function for reuse and testability.

Note: In a real CI script (not a Jupyter notebook) `sys.exit(1)` terminates the process. In a notebook we raise an exception instead to demonstrate the failure path without killing the kernel.

In [ ]:
def ci_gate(checkpoint_result, notebook_mode: bool = True) -> None:
    """Fail the CI job (or raise) if any suite in the Checkpoint failed.

    Args:
        checkpoint_result: CheckpointResult returned by checkpoint.run().
        notebook_mode:     If True, raise RuntimeError instead of sys.exit(1)
                           so the notebook kernel keeps running.

    Raises:
        RuntimeError: (notebook_mode=True) when validation failed.
        SystemExit:   (notebook_mode=False) with code 1 when validation failed.
    """
    if not checkpoint_result.success:
        failed_suites = [
            identifier.expectation_suite_identifier.name
            for identifier, vr in checkpoint_result.run_results.items()
            if not vr.success
        ]
        msg = f'CI gate FAILED — suites with failures: {failed_suites}'
        if notebook_mode:
            raise RuntimeError(msg)
        else:
            print(msg, file=sys.stderr)
            sys.exit(1)
    print('CI gate PASSED — all suites green')


# Test on clean data — should pass
ci_gate(checkpoint_result)

In [ ]:
# Demonstrate gate firing on bad data
bad_row = pd.DataFrame([{
    'order_id':    'ORD-99999',
    'customer_id': 1234,
    'amount':      None,        # <-- null violates critical suite
    'status':      'UNKNOWN',   # <-- not in value_set: violates distributions suite
    'email':       'not-email',
}])
dirty_df = pd.concat([orders_df, bad_row], ignore_index=True)

dirty_result = checkpoint.run(batch_parameters={'dataframe': dirty_df})
print('Dirty CheckpointResult.success:', dirty_result.success)

try:
    ci_gate(dirty_result)
except RuntimeError as exc:
    print(f'Gate fired as expected: {exc}')

## 6. Data Docs Customization

Read the [Data Docs customization guide](https://docs.greatexpectations.io/docs/core/configure_project_settings/configure_data_docs_sites/).

Key customization options in `great_expectations.yml`:

```yaml
data_docs_sites:
  local_site:
    class_name: SiteBuilder
    show_how_to_buttons: true
    store_backend:
      class_name: TupleFilesystemStoreBackend
      base_directory: uncommitted/data_docs/local_site/
    site_index_builder:
      class_name: DefaultSiteIndexBuilder
```

You can configure:
- **Multiple sites** — e.g., a local HTML site and an S3-hosted site simultaneously.
- **Custom renderers** — swap out how expectations and results render in HTML.
- **`show_how_to_buttons`** — hide/show inline GX tutorial links.
- **S3/GCS backends** — point `store_backend` at a cloud bucket for shared team visibility.

Below we print the default site config accessible from the EphemeralDataContext.

In [ ]:
import json as _json

# Serialize context config to inspect Data Docs site settings
try:
    config_dict = _json.loads(context.config.json())
    docs_config = config_dict.get('data_docs_sites', {})
    print('Data Docs sites in current context config:')
    print(_json.dumps(docs_config, indent=2))
except Exception as e:
    print(f'Config introspection note: {e}')
    print('Default site: local_site (TupleFilesystemStoreBackend in ephemeral temp dir)')

## Challenge — Multi-Suite Checkpoint with Failure Reporting

1. Create a third suite called `orders.schema` that checks column count equals 5 and that `order_id` values are unique.
2. Add it to the existing Checkpoint (or create a new Checkpoint) alongside the two existing suites.
3. Inject a duplicate `order_id` into the DataFrame.
4. Run the Checkpoint, use `ci_gate()` to catch the failure, and print a per-suite failure report.

In [ ]:
# ---- Challenge solution ----

# 1. Create orders.schema suite
suite_schema = context.suites.add(gx.ExpectationSuite(name='orders.schema'))
suite_schema.add_expectation(gx.expectations.ExpectTableColumnCountToEqual(value=5))
suite_schema.add_expectation(
    gx.expectations.ExpectColumnValuesToBeUnique(column='order_id')
)
context.suites.save(suite_schema)
print('orders.schema created with', len(suite_schema.expectations), 'expectations')

# 2. Create a new ValidationDefinition and Checkpoint with all three suites
vd_schema = context.validation_definitions.add(
    gx.ValidationDefinition(
        name='vd_orders_schema',
        data=batch_def,
        suite=context.suites.get('orders.schema'),
    )
)

full_checkpoint = context.checkpoints.add(
    gx.Checkpoint(
        name='orders_full_checkpoint',
        validation_definitions=[vd_critical, vd_dist, vd_schema],
        actions=[update_docs_action],
        result_format={'result_format': 'COMPLETE'},
    )
)
print('Full Checkpoint created with', len(full_checkpoint.validation_definitions), 'validation definitions')

# 3. Inject a duplicate order_id
dup_row = orders_df.iloc[0:1].copy()  # duplicate row 0
dup_df = pd.concat([orders_df, dup_row], ignore_index=True)
print(f'DataFrame with duplicate: {len(dup_df)} rows (original: {len(orders_df)})')

# 4. Run and report
dup_result = full_checkpoint.run(batch_parameters={'dataframe': dup_df})
print('\nCheckpointResult.success:', dup_result.success)
print('\nPer-suite report:')
for identifier, vr in dup_result.run_results.items():
    suite_name = identifier.expectation_suite_identifier.name
    failed_exps = [r.expectation_config.type for r in vr.results if not r.success]
    status = 'PASS' if vr.success else f'FAIL ({len(failed_exps)} expectations)'
    print(f'  {suite_name:35s} {status}')
    for exp_type in failed_exps:
        print(f'      - {exp_type}')

try:
    ci_gate(dup_result)
except RuntimeError as exc:
    print(f'\nGate fired: {exc}')

## Recap

| Concept | Key API |
|---|---|
| Pair suite + batch | `gx.ValidationDefinition(name, data, suite)` |
| Orchestrate many suites | `gx.Checkpoint(name, validation_definitions, actions)` |
| Run the Checkpoint | `checkpoint.run(batch_parameters=...)` |
| Aggregate result | `checkpoint_result.success` |
| Per-suite results | `checkpoint_result.run_results.items()` |
| Trigger Data Docs | `gx.checkpoint.UpdateDataDocsAction(name=...)` |
| CI gate | `if not result.success: sys.exit(1)` |

**Key takeaways:**

- A Checkpoint is a composition of ValidationDefinitions — not a replacement.
- `UpdateDataDocsAction` is cheap: always include it so stakeholders can browse HTML reports.
- The CI gate is intentionally simple — complexity belongs in the suite, not the gate.

**Next up — Day 6:** Embedding Checkpoints into a four-step pipeline and detecting schema drift.